# Full dataset Labeling and parameter caching

Notebook9 — canonical SemanticSchedulerEnd2End source, shared `parameters` imports.

Pipeline: image -> selected VLM -> offline image-only SPI labels; image -> one
YOLO-World OR Grounding DINO -> count/category/box extents; sharedCLIP -> relevance
and causal perUE novelty. Post-text priority/native truth remain evaluator-only.

**Default is an executed synthetic walkthrough, not a real300/full-dataset run.**
Notebook8 winner review and real SPI readiness are pending. No arbitrary winner,
three-class fallback, paid service, bulk download or hidden real run. Runtime
receipts print actual environment; intended local environment is project's.venv
(requirements/lock in canonical root). On Kaggle use the managed runtime and
model-specific pinned dependencies; a local venv does not transfer.

Run from project root or set SEMANTIC_PROJECT_ROOT. Parameter modules contain
production logic; notebook only orchestrates. New outputs use unique run IDs.
Do not upload as a new Kaggle identity: stage the next version of the existing
notebook only after notebook8 review and source packaging/preflight.

The dummy readiness receipt is marked fixture_only and is never a real release.
Real training needs accepted image-only teacher labels, sufficient all4class
support and grouped fitting; real parameter cache requires accepted classifier
and readiness receipt. Current cache refuses overwrite; partial shards/checksums
survive failure, but automatic resume and full FLOP accounting are not implemented.


In [ ]:
import os,sys,json,time,hashlib,subprocess,importlib.metadata as metadata
from pathlib import Path
print('START notebook9 setup',flush=True)
ROOT=Path(os.environ.get('SEMANTIC_PROJECT_ROOT',Path.cwd())).resolve()
if not (ROOT/'parameters/dataset_pipeline.py').is_file():raise RuntimeError('Set SEMANTIC_PROJECT_ROOT to canonical project checkout')
sys.path.insert(0,str(ROOT))
print('Repository',ROOT,'Python',sys.executable,sys.version,flush=True)
print('Revision',subprocess.check_output(['git','rev-parse','HEAD'],cwd=ROOT,text=True).strip(),flush=True)
print('Packages',{p:metadata.version(p) for p in ('numpy','Pillow','nbformat')},flush=True)
try:
 import torch
 print('Torch',torch.__version__,'CUDA',torch.version.cuda,'GPUs',[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())],flush=True)
except ImportError:print('Torch unavailable; real neural path blocked')
RUN_ID=os.environ.get('PARAMETER_RUN_ID',time.strftime('%Y%m%d_%H%M%S'))
OUTPUT=ROOT/'experiments/parameter_finalization/runs'/('notebook9_'+RUN_ID)
OUTPUT.mkdir(parents=True,exist_ok=False)
RUN_REAL=False
print('Output',OUTPUT,'RUN_REAL',RUN_REAL,flush=True)
print('END notebook9 setup',flush=True)


In [ ]:
print('START synthetic shared-pipeline walkthrough',flush=True)
from experiments.parameter_finalization.test_dataset_pipeline import run_dummy
DUMMY_OUTPUT=run_dummy(OUTPUT/'dummy')
print('Dummy status',json.loads((DUMMY_OUTPUT/'status.json').read_text()),flush=True)
print('SYNTHETIC FIXTURE ONLY: no model/dataset quality, actual GPU or measured generalization claim',flush=True)
print('END synthetic walkthrough',flush=True)


In [ ]:
print('START real-run contract',flush=True)
# Supply a JSON config only after notebook8 review. No model guessed automatically.
CONFIG_PATH=Path(os.environ.get('PARAMETER_CONFIG','notebook9_config.json'))
CONFIG=json.loads(CONFIG_PATH.read_text()) if CONFIG_PATH.is_file() else None
if RUN_REAL and CONFIG is None:raise RuntimeError('Missing reviewed config; no downloads or full run')
print('Config present',CONFIG is not None,'Real execution enabled',RUN_REAL,flush=True)
print('Expected fields: records_path, selection_receipt, teacher_key, features_path, model_path, readiness_path, clip_checkpoint, detector, detector_checkpoint',flush=True)
print('END real-run contract',flush=True)


In [ ]:
print('START selected-teacher labeling / SPI training-data export',flush=True)
if RUN_REAL:
 from parameters.dataset_pipeline import label_dataset,export_spi_training
 from parameters.dataset_pipeline import load_selected_teacher
 records=json.loads(Path(CONFIG['records_path']).read_text())
 receipt=json.loads(Path(CONFIG['selection_receipt']).read_text())
 teacher=load_selected_teacher(receipt)
 try:label_dataset(records,teacher,receipt,OUTPUT/'teacher_labels')
 finally:teacher.close()
 # Existing feature cache is identity-checked by the training exporter.
 export_spi_training(CONFIG['features_path'],OUTPUT/'teacher_labels/labels.json',OUTPUT/'spi_training')
 print('Training inputs ready; fitting uses existing grouped spi_model_selection after support review',flush=True)
else:print('BLOCKED: no selected teacher yet. No real image labeling or training data created.',flush=True)
print('END selected-teacher stage',flush=True)


In [ ]:
print('START readiness-gated parameter cache',flush=True)
if RUN_REAL:
 from parameters.readiness import require_spi_ready
 from parameters.dataset_pipeline import cache_parameters
 from parameters.minimal_vision import MinimalVisionProvider
 from parameters.contracts import MissionSpec
 from parameters.scene_measurements import YOLOWorldSceneProvider
 from parameters.grounding_scene import GroundingDINOSceneProvider
 require_spi_ready(CONFIG['model_path'],CONFIG['readiness_path'])
 p=json.loads((ROOT/'parameters/config/ue_parameter_profiles_v2.json').read_text())
 profiles=[(r['ue_id'],MissionSpec(r['mission_id'],r['paragraph'],((r['mission_id'],r['paragraph']),),allowed_delay_ms=r['allowed_delay_ms'])) for r in p['profiles']]
 provider=MinimalVisionProvider(CONFIG['clip_checkpoint'],{m.mission_id:m for _,m in profiles},device=CONFIG.get('device','cpu'))
 if CONFIG['detector']=='yolo_world':scene=YOLOWorldSceneProvider(CONFIG['detector_checkpoint'],device=CONFIG.get('device','cpu'))
 elif CONFIG['detector']=='grounding_dino':scene=GroundingDINOSceneProvider(CONFIG['detector_checkpoint'],device=CONFIG.get('device','cpu'))
 else:raise ValueError('Choose exactly one detector')
 try:cache_parameters(records,profiles,provider,scene,CONFIG['model_path'],CONFIG['readiness_path'],OUTPUT/'parameter_cache')
 finally:
  provider.close()
  if hasattr(scene,'close'):scene.close()
else:print('BLOCKED: real four-class SPI readiness is pending; dummy cache is separate.',flush=True)
print('END parameter-cache stage',flush=True)


In [ ]:
print('START full-corpus resource gate',flush=True)
print('Full sources: LADI-v2, MEDIC, CrisisMMD and legitimately linked TREC-IS images.',flush=True)
print('Do not invent inaccessible assets or image urgency labels from post priority.',flush=True)
print('Required: deduplicated grouped manifest, source licenses/revisions/native splits, projected bytes and GPU hours; <=20GB data/eight aggregate GPU hours.',flush=True)
print('Full-corpus extraction is NOT launched by this notebook before the reviewed300 milestone and resource receipt.',flush=True)
print('Timing: per-stage calls and cold-first status. FLOPs: null unless separately supported-operator profiled; not complete totals.',flush=True)
print('END full-corpus resource gate',flush=True)
